# Deep Learning Lizard Challenge - score boost

Deze notebook volgt de PowerPoint: EDA, eigen CNN, transfer learning, data augmentation, training plots, confusion matrix en een geldige Kaggle submission.

Score-focus: sterker EfficientNetV2-model, fine-tuning met lage learning rate, validatiegestuurde modelkeuze en TTA voor stabielere Kaggle-voorspellingen.

## Score boost 384 + AdamW

Aanpassingen in deze kopie:
- gebruikt de nieuwe `train.zip`-beelden in `lizard-prediction-thomas-more/train`
- filtert `train.csv`-rijen waarvan het beeld niet meer in de zip zit en voegt eventuele extra beelden uit klassmappen toe
- traint op 384x384 in plaats van 300x300
- gebruikt sterkere Keras data augmentation-lagen, geen mixup/cutmix
- gebruikt AdamW in plaats van Adam
- gebruikt twee Dense-lagen met GELU in de transfer-learning head
- houdt de bestaande voorzichtige crop-review stap voor Green/Brown Anole, maar zonder validatie-leakage
- traint ook een eigen CNN-baseline, zoals de PowerPoint als minimumvereiste vraagt
- rapporteert apart de verwarring tussen Green Anole en Brown Anole

**PowerPoint-check:** deze notebook bevat markdown-structuur, EDA, een eigen Keras/TensorFlow CNN, transfer learning, data augmentation, training plots met validatieset, evaluatie met confusion matrix, een geldige Kaggle-submission en een GenAI-vermelding.

**Fixed copy:** oude outputs gewist en `GaussianNoise(0.02)` expliciet gezet. Run de notebook volledig opnieuw voordat je hem als report uploadt, zodat de plots en confusion matrix zichtbaar in de output staan.


In [ ]:
from pathlib import Path
import random, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__)

## 1. Data inlezen

In [ ]:
BASE_DIR = Path("lizard-prediction-thomas-more")
TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"

train_df = pd.read_csv(BASE_DIR / "train.csv")
test_df = pd.read_csv(BASE_DIR / "test.csv")
sample_sub = pd.read_csv(BASE_DIR / "sample_submission.csv")

CLASS_NAMES = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
print(len(train_df), "trainbeelden")
print(len(test_df), "testbeelden")
print(NUM_CLASSES, "klassen")
for i, name in enumerate(CLASS_NAMES):
    print(i, name)
train_df.head()

In [ ]:
def build_train_file_index(train_dir):
    index = {}
    for class_dir in sorted([p for p in train_dir.iterdir() if p.is_dir()]):
        files = {}
        for path in class_dir.iterdir():
            if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                files[path.name] = path
                files[path.stem] = path
        index[class_dir.name] = files
    return index

TRAIN_FILE_INDEX = build_train_file_index(TRAIN_DIR)

def resolve_train_path(row):
    label = int(row["label"])
    class_name = CLASS_NAMES[label]
    raw_id = str(row["id"])
    candidates = [raw_id, f"{raw_id}.jpg", f"{raw_id}.jpeg", f"{raw_id}.png", Path(raw_id).stem]
    for candidate in candidates:
        path = TRAIN_FILE_INDEX.get(class_name, {}).get(candidate)
        if path is not None and path.exists():
            return str(path)
    return None

def resolve_test_path(image_id):
    for ext in [".jpg", ".jpeg", ".png"]:
        path = TEST_DIR / f"{int(image_id)}{ext}"
        if path.exists():
            return str(path)
    raise FileNotFoundError(image_id)

train_df["path"] = train_df.apply(resolve_train_path, axis=1)
missing_df = train_df[train_df["path"].isna()].copy()
if len(missing_df):
    missing_df["class_name"] = missing_df["label"].map(lambda i: CLASS_NAMES[int(i)])
    print(f"{len(missing_df)} train.csv-rijen overgeslagen omdat de zip deze beelden niet bevat.")
    display(missing_df[["id", "label", "class_name"]].head(50))
train_df = train_df.dropna(subset=["path"]).copy()

known_paths = set(train_df["path"].map(lambda p: str(Path(p))))
extra_rows = []
for label, class_name in enumerate(CLASS_NAMES):
    class_dir = TRAIN_DIR / class_name
    for path in sorted(class_dir.iterdir()):
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            path_str = str(path)
            if path_str not in known_paths:
                extra_rows.append({"id": path.name, "label": label, "path": path_str, "source": "extra_from_folder"})
if extra_rows:
    print(f"{len(extra_rows)} extra beelden uit de zip toegevoegd op basis van de klassmap.")
    display(pd.DataFrame(extra_rows).head(20))
    train_df = pd.concat([train_df, pd.DataFrame(extra_rows)], ignore_index=True)

test_df["path"] = test_df["id"].apply(resolve_test_path)
assert train_df["path"].map(lambda p: Path(p).exists()).all()
assert test_df["path"].map(lambda p: Path(p).exists()).all()
print(len(train_df), "bruikbare trainbeelden")
print("Alle beschikbare image paths gevonden")


## 2. EDA

De klassen zijn licht uit balans. Daarom gebruiken we later `class_weight`.

In [ ]:
class_counts = train_df["label"].value_counts().sort_index()
eda = pd.DataFrame({"label": class_counts.index, "class_name": [CLASS_NAMES[i] for i in class_counts.index], "count": class_counts.values})
display(eda)

plt.figure(figsize=(10,4))
plt.bar(eda["class_name"], eda["count"])
plt.xticks(rotation=35, ha="right")
plt.title("Klasseverdeling")
plt.ylabel("Aantal afbeeldingen")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,7))
for label in range(NUM_CLASSES):
    row = train_df[train_df["label"] == label].sample(1, random_state=SEED).iloc[0]
    img = keras.utils.load_img(row["path"], target_size=(180, 180))
    plt.subplot(2, 4, label + 1)
    plt.imshow(img)
    plt.title(f"{label}: {CLASS_NAMES[label]}", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Stratified split en class weights

In [ ]:
train_split, val_split = train_test_split(train_df, test_size=0.20, random_state=SEED, stratify=train_df["label"])
print("train", len(train_split), "validatie", len(val_split))
print(val_split["label"].value_counts().sort_index())


## 3b. Veilige extra crops voor Green/Brown Anole

We voegen cropped kopieen pas toe na de stratified split. Zo kan een originele afbeelding niet in validatie zitten terwijl een crop ervan in training zit. De code hieronder werkt alleen op `Green_anole` en `Brown_anole`, bewaart de originele beelden, en toont twijfelgevallen apart voor manuele controle.


In [ ]:
TARGET_CROP_CLASSES = ["Green_anole", "Brown_anole"]
CROPPED_SUFFIX = "_cropped"

# Deze lijst bevat alleen beelden die visueel verdacht zijn: kleine hagedis,
# veel achtergrond, slechte centrering of duidelijke afleiding.
SUSPECT_CROP_FILES = {
    "Brown_anole": [
        "Brown_anole_Brown_anole_104.jpg",
        "Brown_anole_Brown_anole_126.jpg",
        "Brown_anole_Brown_anole_129.jpg",
        "Brown_anole_Brown_anole_139.jpg",
        "Brown_anole_Brown_anole_156.jpg",
        "Brown_anole_Brown_anole_176.jpg",
        "Brown_anole_Brown_anole_180.jpg",
    ],
    "Green_anole": [
        "Green_anole_Green_anole_1.jpg",
        "Green_anole_Green_anole_114.jpg",
        "Green_anole_Green_anole_117.jpg",
        "Green_anole_Green_anole_122.jpg",
        "Green_anole_Green_anole_141.jpg",
        "Green_anole_Green_anole_153.jpg",
        "Green_anole_Green_anole_161.jpg",
        "Green_anole_Green_anole_175.jpg",
        "Green_anole_Green_anole_212.jpg",
        "Green_anole_Green_anole_219.jpg",
        "Green_anole_Green_anole_45.jpg",
    ],
}

# Semi-automatisch en veilig: alleen deze duidelijke cases krijgen een handmatig
# gecontroleerde bounding box. Alles zonder box wordt getoond als manueel te checken.
# Box-formaat: (left, top, right, bottom) in pixels van de originele afbeelding.
SAFE_CROP_BOXES = {
    "Brown_anole_Brown_anole_129.jpg": (450, 650, 950, 1250),
    "Brown_anole_Brown_anole_139.jpg": (600, 950, 1150, 1750),
    "Brown_anole_Brown_anole_176.jpg": (620, 1050, 1100, 1700),
    "Green_anole_Green_anole_1.jpg": (420, 0, 1400, 1900),
    "Green_anole_Green_anole_114.jpg": (500, 850, 1450, 1500),
    "Green_anole_Green_anole_122.jpg": (75, 135, 190, 290),
    "Green_anole_Green_anole_141.jpg": (380, 250, 650, 560),
    "Green_anole_Green_anole_153.jpg": (450, 650, 1080, 1300),
    "Green_anole_Green_anole_161.jpg": (230, 300, 1100, 1050),
    "Green_anole_Green_anole_175.jpg": (850, 850, 1500, 1350),
    "Green_anole_Green_anole_212.jpg": (430, 400, 1000, 1300),
    "Green_anole_Green_anole_219.jpg": (600, 500, 1550, 1180),
}

def cropped_name(image_id):
    stem = Path(image_id).stem
    ext = Path(image_id).suffix or ".jpg"
    return f"{stem}{CROPPED_SUFFIX}{ext}"

def clamp_box(box, width, height):
    left, top, right, bottom = box
    left = max(0, min(int(left), width - 1))
    top = max(0, min(int(top), height - 1))
    right = max(left + 1, min(int(right), width))
    bottom = max(top + 1, min(int(bottom), height))
    return left, top, right, bottom

def is_safe_crop(box, width, height):
    # Veiligheidscheck: crop mag niet extreem klein zijn en moet echt achtergrond verwijderen.
    left, top, right, bottom = box
    crop_area = (right - left) * (bottom - top)
    image_area = width * height
    crop_ratio = crop_area / image_area
    min_side_ok = (right - left) >= 90 and (bottom - top) >= 90
    useful_crop = 0.03 <= crop_ratio <= 0.75
    return min_side_ok and useful_crop

def make_cropped_copy(row, box):
    src = Path(row["path"])
    dst = src.with_name(cropped_name(row["id"]))
    with Image.open(src) as image:
        image = image.convert("RGB")
        box = clamp_box(box, image.width, image.height)
        if not is_safe_crop(box, image.width, image.height):
            return None, "box afgekeurd door veiligheidscheck"
        crop = image.crop(box)
        crop.save(dst, quality=95)
    return str(dst), None

# Alleen deze klassen worden gecontroleerd.
target_labels = [CLASS_NAMES.index(name) for name in TARGET_CROP_CLASSES]
target_df = train_df[train_df["label"].isin(target_labels)].copy()
checked_counts = target_df.assign(class_name=target_df["label"].map(lambda i: CLASS_NAMES[int(i)]))["class_name"].value_counts()

# Geen leakage: crop alleen bronnen die in train_split zitten; val_split blijft ongewijzigd.
train_ids = set(train_split["id"])
val_ids = set(val_split["id"])
new_rows = []
crop_log = []
manual_review_rows = []

for class_name, filenames in SUSPECT_CROP_FILES.items():
    for image_id in filenames:
        row_match = train_df[train_df["id"] == image_id]
        if row_match.empty:
            manual_review_rows.append({"class": class_name, "id": image_id, "reden": "niet gevonden in train.csv"})
            continue
        row = row_match.iloc[0]
        if image_id in val_ids:
            manual_review_rows.append({"class": class_name, "id": image_id, "reden": "zit in validatie; niet croppen om leakage te vermijden"})
            continue
        if image_id not in train_ids:
            manual_review_rows.append({"class": class_name, "id": image_id, "reden": "niet in train_split gevonden"})
            continue
        if image_id not in SAFE_CROP_BOXES:
            manual_review_rows.append({"class": class_name, "id": image_id, "reden": "verdacht, maar geen betrouwbare crop-box"})
            continue

        cropped_path, error = make_cropped_copy(row, SAFE_CROP_BOXES[image_id])
        if error:
            manual_review_rows.append({"class": class_name, "id": image_id, "reden": error})
            continue

        crop_id = cropped_name(image_id)
        new_rows.append({"id": crop_id, "label": int(row["label"]), "path": cropped_path})
        crop_log.append({"class": class_name, "original": image_id, "cropped": crop_id, "path": cropped_path})

if new_rows:
    crop_df = pd.DataFrame(new_rows)
    # Voorkom dubbele rijen wanneer je de cel opnieuw uitvoert.
    train_split = pd.concat([train_split, crop_df], ignore_index=True).drop_duplicates(subset=["path"]).reset_index(drop=True)
else:
    crop_df = pd.DataFrame(columns=["id", "label", "path"])

crop_summary = pd.DataFrame({
    "klasse": TARGET_CROP_CLASSES,
    "gecontroleerd": [int(checked_counts.get(name, 0)) for name in TARGET_CROP_CLASSES],
    "gecropt": [sum(item["class"] == name for item in crop_log) for name in TARGET_CROP_CLASSES],
})

print("Crop summary")
display(crop_summary)
print("Gecropte bestanden")
display(pd.DataFrame(crop_log))
print("Manueel controleren / niet automatisch gecropt")
display(pd.DataFrame(manual_review_rows))


In [ ]:
def show_crop_examples(crop_log, max_examples=8):
    if not crop_log:
        print("Geen crops gemaakt.")
        return
    examples = crop_log[:max_examples]
    fig, axes = plt.subplots(len(examples), 2, figsize=(8, 3 * len(examples)))
    if len(examples) == 1:
        axes = np.array([axes])
    for ax_row, item in zip(axes, examples):
        original_path = TRAIN_DIR / item["class"] / item["original"]
        cropped_path = Path(item["path"])
        for ax, image_path, title in [
            (ax_row[0], original_path, "origineel"),
            (ax_row[1], cropped_path, "cropped"),
        ]:
            ax.imshow(Image.open(image_path))
            ax.set_title(f"{title}\n{image_path.name}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()

def show_manual_review_images(manual_review_rows, max_images=12):
    review_df = pd.DataFrame(manual_review_rows)
    if review_df.empty:
        print("Geen manuele review nodig.")
        return
    # Toon alleen beelden die echt bestaan; validatiebeelden worden ook getoond, maar niet gecropt.
    existing = []
    for _, row in review_df.iterrows():
        path = TRAIN_DIR / row["class"] / row["id"]
        if path.exists():
            existing.append((path, row["reden"]))
    if not existing:
        print("Geen bestaande reviewbeelden gevonden.")
        return
    existing = existing[:max_images]
    cols = 3
    rows = int(np.ceil(len(existing) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, (image_path, reason) in zip(axes, existing):
        ax.imshow(Image.open(image_path))
        ax.set_title(f"{image_path.name}\n{reason}", fontsize=9)
        ax.axis("off")
    for ax in axes[len(existing):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_crop_examples(crop_log)
show_manual_review_images(manual_review_rows)


In [ ]:
weights = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES), y=train_split["label"].values)
CLASS_WEIGHTS = {i: float(w) for i, w in enumerate(weights)}
CLASS_WEIGHTS


## 4. TensorFlow datasets

EfficientNetV2 met `include_preprocessing=True` verwacht RGB-pixels in `[0,255]`.

In [ ]:
BATCH_SIZE = 8          # 384x384 vraagt meer geheugen dan 300x300
IMG_SIZE = (384, 384)   # hoger detailniveau; meestal beter voor fijne soortverschillen
BACKBONE = "EfficientNetV2S"  # alternatief: "EfficientNetV2M" voor een zwaardere run
AUTOTUNE = tf.data.AUTOTUNE

# Geen mixup/cutmix: voor deze challenge houden we labels zuiver, vooral bij Brown/Green Anole.
def load_image(path, label=None, img_size=IMG_SIZE):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.convert_image_dtype(image, tf.float32) * 255.0
    image = tf.image.resize(image, img_size)
    if label is None:
        return image
    return image, tf.cast(label, tf.int32)

def make_labeled_ds(df, img_size=IMG_SIZE, shuffle=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label"].values))
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: load_image(p, y, img_size), num_parallel_calls=AUTOTUNE)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

def make_unlabeled_ds(paths, img_size=IMG_SIZE, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(np.array(paths, dtype=str))
    ds = ds.map(lambda p: load_image(p, None, img_size), num_parallel_calls=AUTOTUNE)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_labeled_ds(train_split, shuffle=True)
val_ds = make_labeled_ds(val_split)


## 5. Eigen CNN baseline

Deze baseline is vooral voor de minimumvereiste uit de opdracht. Het transfer-learning model zal normaal beter scoren.

In [ ]:
def build_own_cnn(input_shape, num_classes):
    return keras.Sequential([
        keras.Input(shape=input_shape),
        layers.Rescaling(1./255),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.15),
        layers.Conv2D(32, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(256, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.GlobalAveragePooling2D(),
        layers.Dropout(0.45),
        layers.Dense(256, activation="gelu"),
        layers.BatchNormalization(),
        layers.Dropout(0.35),
        layers.Dense(96, activation="gelu"),
        layers.Dropout(0.25),
        layers.Dense(num_classes, activation="softmax"),
    ], name="own_cnn_baseline")

cnn_model = build_own_cnn((*IMG_SIZE, 3), NUM_CLASSES)
cnn_model.compile(optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn_model.summary()


In [ ]:
TRAIN_OWN_CNN = True
if TRAIN_OWN_CNN:
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint("scoreboost_own_cnn.keras", monitor="val_accuracy", save_best_only=True),
    ]
    history_cnn = cnn_model.fit(train_ds, validation_data=val_ds, epochs=35, class_weight=CLASS_WEIGHTS, callbacks=callbacks)
else:
    history_cnn = None
    print("Eigen CNN niet opnieuw getraind. Zet TRAIN_OWN_CNN=True als je dit wilt runnen.")

## 6. Transfer learning + fine-tuning

In [ ]:
def backbone_class(name):
    if name == "EfficientNetV2S":
        return keras.applications.EfficientNetV2S
    if name == "EfficientNetV2M":
        return keras.applications.EfficientNetV2M
    raise ValueError("Gebruik EfficientNetV2S of EfficientNetV2M")

def build_transfer_model(backbone_name=BACKBONE, img_size=IMG_SIZE):
    Backbone = backbone_class(backbone_name)
    base = Backbone(include_top=False, weights="imagenet", input_shape=(*img_size, 3), include_preprocessing=True)
    base.trainable = False

    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.12),
        layers.RandomZoom((-0.10, 0.18)),
        layers.RandomTranslation(0.08, 0.08),
        layers.RandomContrast(0.22),
        layers.RandomBrightness(0.12, value_range=(0, 255)),
        layers.GaussianNoise(0.02),  # Keras vereist stddev tussen 0 en 1
    ], name="data_augmentation")

    inputs = keras.Input(shape=(*img_size, 3), name="image")
    x = augmentation(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(512, activation="gelu", kernel_regularizer=keras.regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(192, activation="gelu", kernel_regularizer=keras.regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="species")(x)
    return keras.Model(inputs, outputs, name=f"lizard_{backbone_name}_384_adamw"), base

model, base_model = build_transfer_model()
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


In [ ]:
TRAIN_TRANSFER = True
PHASE1_MODEL = "scoreboost_384_phase1.keras"
FINETUNED_MODEL = "scoreboost_384_finetuned.keras"

if TRAIN_TRANSFER:
    callbacks1 = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.35, patience=3, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint(PHASE1_MODEL, monitor="val_accuracy", save_best_only=True),
    ]
    history_phase1 = model.fit(train_ds, validation_data=val_ds, epochs=25, class_weight=CLASS_WEIGHTS, callbacks=callbacks1)
else:
    history_phase1 = None


In [ ]:
if TRAIN_TRANSFER:
    model = keras.models.load_model(PHASE1_MODEL)
    backbone = next(layer for layer in model.layers if "efficientnet" in layer.name.lower())
    backbone.trainable = True
    fine_tune_last_n = 80 if BACKBONE == "EfficientNetV2S" else 110
    for layer in backbone.layers[:-fine_tune_last_n]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    lr_schedule = keras.optimizers.schedules.CosineDecay(6e-6, decay_steps=max(1, len(train_split)//BATCH_SIZE)*40, alpha=0.08)
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=5e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    callbacks2 = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(FINETUNED_MODEL, monitor="val_accuracy", save_best_only=True),
    ]
    history_phase2 = model.fit(train_ds, validation_data=val_ds, epochs=40, class_weight=CLASS_WEIGHTS, callbacks=callbacks2)
else:
    history_phase2 = None


## 7. Training curves

In [ ]:
def plot_history(history, title):
    if history is None:
        print("Geen history voor", title)
        return
    hist = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    axes[0].plot(hist["accuracy"], label="train")
    axes[0].plot(hist["val_accuracy"], label="validatie")
    axes[0].set_title(title + " accuracy")
    axes[0].legend()
    axes[1].plot(hist["loss"], label="train")
    axes[1].plot(hist["val_loss"], label="validatie")
    axes[1].set_title(title + " loss")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_cnn, "Eigen CNN")
plot_history(history_phase1, "Transfer fase 1")
plot_history(history_phase2, "Fine-tuning")

## 8. Modelkeuze en confusion matrix

We kiezen niet blind een ensemble. Als een ensemble op validatie slechter is dan het beste losse model, gebruiken we het beste losse model. Zo maken we de Kaggle-submission minder fragiel.

In [ ]:
# Fix: lees de echte input-size uit elk opgeslagen .keras model.
def model_input_size(model_path, fallback_img_size):
    m = keras.models.load_model(model_path, compile=False)
    try:
        input_shape = m.input_shape[0] if isinstance(m.input_shape, list) else m.input_shape
        height, width = input_shape[1], input_shape[2]
        if height is None or width is None:
            return tuple(fallback_img_size)
        return (int(height), int(width))
    finally:
        del m
        tf.keras.backend.clear_session(); gc.collect()

def predict_model(model_path, paths, img_size):
    img_size = model_input_size(model_path, img_size)
    m = keras.models.load_model(model_path, compile=False)
    probs = m.predict(make_unlabeled_ds(paths, img_size=img_size), verbose=0)
    del m
    tf.keras.backend.clear_session(); gc.collect()
    return probs

candidate_models = []
for item in [(FINETUNED_MODEL, IMG_SIZE), (PHASE1_MODEL, IMG_SIZE)]:
    if Path(item[0]).exists():
        candidate_models.append(item)

legacy_models = [
    ("scoreboost_finetuned.keras", (300,300)),
    ("scoreboost_phase1.keras", (300,300)),
    ("best_efficientnetv2s_pwp.keras", (260,260)),
    ("phase1_best_efficientnetv2s_pwp.keras", (260,260)),
    ("best_model_fase2.keras", (224,224)),
    ("best_model_fase1.keras", (224,224)),
]

for item in legacy_models:
    if Path(item[0]).exists():
        candidate_models.append(item)

candidate_models = [(path, model_input_size(path, size)) for path, size in candidate_models]
seen=set(); candidate_models=[x for x in candidate_models if not (x[0] in seen or seen.add(x[0]))]
print(candidate_models)


In [ ]:
if not candidate_models:
    raise FileNotFoundError("Geen kandidaatmodellen gevonden. Run eerst de trainingscellen of zet TRAIN_TRANSFER=True.")

val_paths = val_split["path"].tolist()
y_true = val_split["label"].to_numpy()
val_probs_list = []
rows = []
for path, size in candidate_models:
    probs = predict_model(path, val_paths, size)
    acc = accuracy_score(y_true, probs.argmax(axis=1))
    val_probs_list.append(probs)
    rows.append({"model": path, "img_size": size, "val_accuracy": acc})

scores = pd.DataFrame(rows).sort_values("val_accuracy", ascending=False)
display(scores)
best_model_path = scores.iloc[0]["model"]
best_img_size = tuple(scores.iloc[0]["img_size"])
best_index = candidate_models.index((best_model_path, best_img_size))
best_val_probs = val_probs_list[best_index]
best_val_acc = scores.iloc[0]["val_accuracy"]

accs = np.array([r["val_accuracy"] for r in rows])
weights = (accs ** 2) / np.sum(accs ** 2)
ensemble_val_probs = sum(w*p for w, p in zip(weights, val_probs_list))
ensemble_val_acc = accuracy_score(y_true, ensemble_val_probs.argmax(axis=1))
USE_ENSEMBLE = ensemble_val_acc > best_val_acc
print("Best single:", best_model_path, round(best_val_acc,4))
print("Ensemble:", round(ensemble_val_acc,4), "use?", USE_ENSEMBLE)

final_val_probs = ensemble_val_probs if USE_ENSEMBLE else best_val_probs
y_pred = final_val_probs.argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8,8))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.tight_layout(); plt.show()

cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
confusions = []
for true_name in CLASS_NAMES:
    for pred_name in CLASS_NAMES:
        if true_name != pred_name:
            confusions.append({"true": true_name, "predicted": pred_name, "count": int(cm_df.loc[true_name, pred_name])})
confusions = pd.DataFrame(confusions).sort_values("count", ascending=False)
print("Meest verwarde klassen")
display(confusions.head(10))


# Specifieke check voor de vaak verwarde Green/Brown Anole-klassen.
green_brown_pairs = {
    ("Green_anole", "Brown_anole"),
    ("Brown_anole", "Green_anole"),
}
green_brown_confusion = confusions[
    confusions.apply(lambda r: (r["true"], r["predicted"]) in green_brown_pairs, axis=1)
].copy()
print("Green/Brown Anole verwarring")
display(green_brown_confusion)
print(
    "Totaal Green<->Brown verwisselingen:",
    int(green_brown_confusion["count"].sum()) if not green_brown_confusion.empty else 0,
)


## 9. Kaggle submission met TTA

In [ ]:
def predict_with_tta(model_path, paths, img_size, num_tta=5):
    m = keras.models.load_model(model_path)
    base_ds = make_unlabeled_ds(paths, img_size=img_size)
    all_probs = []
    for tta_i in range(num_tta):
        parts = []
        for batch in base_ds:
            x = batch
            if tta_i == 1:
                x = tf.image.flip_left_right(batch)
            elif tta_i == 2:
                x = tf.image.adjust_contrast(batch, 1.08)
            elif tta_i == 3:
                x = tf.image.adjust_brightness(batch, 8.0)
            elif tta_i == 4:
                margin_h = max(1, int(img_size[0] * 0.04))
                margin_w = max(1, int(img_size[1] * 0.04))
                cropped = tf.image.crop_to_bounding_box(batch, margin_h, margin_w, img_size[0] - 2*margin_h, img_size[1] - 2*margin_w)
                x = tf.image.resize(cropped, img_size)
            parts.append(m.predict(x, verbose=0))
        all_probs.append(np.concatenate(parts, axis=0))
    tf.keras.backend.clear_session(); gc.collect()
    return np.mean(all_probs, axis=0)

test_paths = test_df["path"].tolist()
NUM_TTA = 5

if USE_ENSEMBLE:
    probs = 0
    for (path, size), weight in zip(candidate_models, weights):
        print("predict", path, "weight", round(float(weight),3))
        probs = probs + weight * predict_with_tta(path, test_paths, size, NUM_TTA)
    output_name = "submission_scoreboost_384_ensemble_tta.csv"
else:
    print("predict beste model", best_model_path)
    probs = predict_with_tta(best_model_path, test_paths, best_img_size, NUM_TTA)
    output_name = "submission_scoreboost_384_bestmodel_tta.csv"

submission = pd.DataFrame({"id": sample_sub["id"], "label": probs.argmax(axis=1).astype(int)})
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)
assert submission["label"].between(0, NUM_CLASSES-1).all()
submission.to_csv(output_name, index=False)
submission.to_csv("submission.csv", index=False)
print("saved", output_name, "and submission.csv")
display(submission.head(10))
print(submission["label"].value_counts().sort_index())


## 10. Conclusie voor de defense

- De eigen CNN toont convolution, pooling/GAP en classificatie uit de les.
- Transfer learning gebruikt een pretrained EfficientNetV2-body en een eigen classifier head.
- Fine-tuning unfreezet enkel de laatste lagen met een lage learning rate, zodat de ImageNet-kennis niet kapot getraind wordt.
- Data augmentation helpt tegen overfitting.
- De confusion matrix toont welke hagedissoorten nog verward worden.
- TTA maakt de testvoorspellingen stabieler voor Kaggle.

## GenAI-vermelding

AI werd gebruikt om de PowerPointvereisten te controleren en om een verbeterde TensorFlow/Keras-notebook te structureren. De prompts vroegen om de bestaande aanpak te verbeteren zonder de opdrachtregels te breken. De beperking is dat een hogere validatiescore geen garantie geeft op een hogere Kaggle-score; de uiteindelijke CSV moet dus op Kaggle getest worden. Ik heb de code nagekeken en kan de gebruikte technieken uitleggen.